# **For this notebook to work, you must execute ALL the cells in the order in which they appear.**


# **Mount your Google Drive and log in to PySpark**

In [ ]:


# Link your Google Drive so that Colab can access it.
# Colab will ask for authorization to access your Google Drive.
from google.colab import drive
drive.mount("/content/drive")

# Create a Spark session with a custom name.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Colab_Session_5") \
.getOrCreate()


Mounted at /content/drive


# **Observe the data frame we are working with**

In [ ]:

from pyspark.sql.types import *

orderSchema = StructType([
   StructField("SalesOrderNumber", StringType()),
   StructField("SalesOrderLineNumber", IntegerType()),
   StructField("OrderDate", DateType()),
   StructField("CustomerName", StringType()),
   StructField("Email", StringType()),
   StructField("Item", StringType()),
   StructField("Quantity", IntegerType()),
   StructField("UnitPrice", FloatType()),
   StructField("Tax", FloatType())
])

df = spark.read.format("csv").schema(orderSchema).load("/content/drive/MyDrive/Prueba_ventas_1.csv")

df.show()   # In fABRIC USE:    display(df)



+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|                NULL|      NULL|     CustomerName|               Email|                Item|    NULL|     NULL|    NULL|
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|         

# **Query using the Spark DataFrame API (DSL)**

In [ ]:
# Grouping and aggregating using the Spark DataFrame API

from pyspark.sql import functions as F

# We apply the transformations and groupings
resultado_df = df.groupBy("CustomerName") \
  .agg(
      F.countDistinct("SalesOrderNumber").alias("#_Of_Orders"),
      F.sum(F.col("Quantity") * F.col("UnitPrice")).alias("TotalPurchased")
  ) \
  .orderBy(F.col("TotalPurchased").desc())

# Show results
resultado_df.show()   # en Fabric use display(resultado_df)


+----------------+-----------+----------------+
|    CustomerName|#_Of_Orders|  TotalPurchased|
+----------------+-----------+----------------+
|   Wyatt Griffin|          1|3578.27001953125|
|     Drew Sharma|          1|3578.27001953125|
|    Claudia Zeng|          1|3578.27001953125|
|      Angela Cox|          1|3578.27001953125|
|       Ivan Rana|          1|3578.27001953125|
|       Peter Jai|          1|3578.27001953125|
|        Tracy Xu|          1|3578.27001953125|
|   Xavier Harris|          1|3578.27001953125|
|       Trisha Li|          1|3578.27001953125|
|  Richard Parker|          1|3578.27001953125|
|     Sharon Nara|          1|3578.27001953125|
| Savannah Morris|          1|3578.27001953125|
| Randall Carlson|          1|3578.27001953125|
|       Alvin Sun|          1|3578.27001953125|
|      Ian Morris|          1|3578.27001953125|
|  Hailey Stewart|          1|3578.27001953125|
|Adriana Gonzalez|          1|3578.27001953125|
|   Christine Pal|          1|3578.27001

# **Query using the SQL engine**



In [ ]:
# 1. Register the DataFrame as a temporary view in Spark
df.createOrReplaceTempView("orders_view")

# 2. Write and execute the query using SQL
resulting_df = spark.sql("""
    SELECT
        CustomerName,
        COUNT(DISTINCT SalesOrderNumber) AS _Of_Orders,
        SUM(Quantity * UnitPrice) AS TotalPurchased
    FROM orders_view
    GROUP BY CustomerName
    ORDER BY TotalPurchased DESC
""")

# 3. Display results
resulting_df.show()          # In Fabric use display(resultado_df)

+----------------+----------+----------------+
|    CustomerName|_Of_Orders|  TotalPurchased|
+----------------+----------+----------------+
|   Wyatt Griffin|         1|3578.27001953125|
|     Drew Sharma|         1|3578.27001953125|
|    Claudia Zeng|         1|3578.27001953125|
|      Angela Cox|         1|3578.27001953125|
|       Ivan Rana|         1|3578.27001953125|
|       Peter Jai|         1|3578.27001953125|
|        Tracy Xu|         1|3578.27001953125|
|   Xavier Harris|         1|3578.27001953125|
|       Trisha Li|         1|3578.27001953125|
|  Richard Parker|         1|3578.27001953125|
|     Sharon Nara|         1|3578.27001953125|
| Savannah Morris|         1|3578.27001953125|
| Randall Carlson|         1|3578.27001953125|
|       Alvin Sun|         1|3578.27001953125|
|      Ian Morris|         1|3578.27001953125|
|  Hailey Stewart|         1|3578.27001953125|
|Adriana Gonzalez|         1|3578.27001953125|
|   Christine Pal|         1|3578.27001953125|
|   Mathew Su

# **Group by year and count the number of sales**

In [ ]:
# Group by year and count the number of sales

from pyspark.sql.functions import *
yearlySales = df.select(year(col("OrderDate")).alias("Year")).groupBy("Year").count().orderBy("Year")
yearlySales.show()    # En Fabric use:   display(yearlySales)

+----+-----+
|Year|count|
+----+-----+
|NULL|    1|
|2019| 1201|
+----+-----+

